# <center>**Source Model**<center>

**Libraries**

In [ ]:
import numpy as np
import mbloodmoon.iros_management as iros
import mbloodmoon as bm

from _IROS_support import _handle_dirpaths

In [ ]:
mask_FITS = "wfm_mask.fits"

skyfield = "GalacticCenter"
#data_FITS = "20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb"
data_FITS = "20250320_galctr_rxte_sax_2-50keV_1ks_sources_cxb_infmaskdet"
#data_FITS = "20250131_galctr_rxte_sax_2-30keV_1ks_sources_cxb"
#data_FITS = "20250430_galctr_rxte_sax_2-50keV_1ks_realmask_infdet_sources_cxb"
#data_FITS = "20250430_galctr_rxte_sax_2-50keV_1ks_opaquemask_realdet_sources_cxb"
#data_FITS = "20250430_galctr_rxte_sax_2-50keV_1ks_thinmask_realdet_sources_cxb"


#skyfield = "Crab"
#data_FITS = "20250227_crab_cxb_2-50keV_1ks"


N_TEST = "test_vignetting"
#N_TEST = "test_vignetting_opposite_cut"

UPX, UPY = 5, 1

mask_path, simul_data, save_path = _handle_dirpaths(
    mask=mask_FITS,
    skyfield=skyfield,
    simul=data_FITS,
    test_name=N_TEST,
)

cam_a = "cam1a"
cam_b = "cam1b"
dataset = "reconstructed"

filepaths = bm.simulation_files(simul_data)
energy_range = (2, 30)
coords = None


from temp_camera import codedmask

wfm = codedmask(mask_path, UPX, UPY)
sdlA = bm.simulation(filepaths[cam_a][dataset], energy_range=energy_range, coords=coords)
sdlB = bm.simulation(filepaths[cam_b][dataset], energy_range=energy_range, coords=coords)

catalogA = bm.simulation(filepaths[cam_a]["sources"])
catalogB = bm.simulation(filepaths[cam_b]["sources"])

In [ ]:
mask_path, simul_data, save_path

In [ ]:
database_name = save_path + f"IROS_sources_database_TEST_{N_TEST}.fits"
database = iros.load_iros_data(database_name)

In [ ]:
import numpy.typing as npt
import matplotlib.pyplot as plt

#from mbloodmoon.mask import _convolution_kernel_psfy

from mbloodmoon.mask import _bisect_interval, psfy_wfm

#def _convolution_kernel_psfy(camera: CodedMaskCamera) -> np.array:
#    """
#    Returns PSF convolution kernel.
#    At present, it ignores the `x` direction, since PSF characteristic lenght is much shorter
#    than typical bin size, even at moderately large upscales.
#
#    Args:
#        camera: a CodedMaskCamera object.
#
#    Returns:
#        A column array convolution kernel.
#    """
#    bins = camera.bins_detector
#    min_bin, max_bin = _bisect_interval(bins.y, -camera.mdl["slit_deltay"], camera.mdl["slit_deltay"])
#    print(camera.mdl["slit_deltay"], min_bin, max_bin)
#    bin_edges = bins.y[min_bin : max_bin + 1]
#    print(len(bin_edges))
#    #midpoints = (bin_edges[1:] + bin_edges[:-1]) / 2
#    #midpoints = bin_edges + 0.5 * camera.mdl["mask_deltay"] / camera.upscale_f.y
#    midpoints = bin_edges
#    kernel = psfy_wfm(midpoints).reshape(len(midpoints), -1)
#    kernel = kernel / np.sum(kernel)
#    return kernel, midpoints


def _convolution_kernel_psfy(camera: CodedMaskCamera) -> npt.NDArray:
    """
    Returns PSF convolution kernel.
    At present, it ignores the `x` direction, since PSF characteristic lenght is much shorter
    than typical bin size, even at moderately large upscales.

    Args:
        camera: a CodedMaskCamera object.

    Returns:
        A column array convolution kernel.
    """
    # we take a whole slit to have a good kernel spatial extension
    slit = camera.mdl["slit_deltay"]
    # to define the kernel, we only need an array with the same binning step
    num = int(2 * slit * camera.upscale_f.y / camera.mdl["mask_deltay"]) + 1
    bins = np.linspace(-slit, slit, num)
    kernel = psfy_wfm(bins).reshape(len(bins), -1)
    kernel = kernel / np.sum(kernel)
    return kernel



cam = codedmask(mask_path, upscale_y=5)

kernel, bins = _convolution_kernel_psfy(cam)
#bins = np.arange(-len(kernel) // 2, len(kernel) // 2) * wfm.mdl["mask_deltay"] / wfm.upscale_f.y
#bins = np.arange(len(kernel)) * wfm.mdl["mask_deltay"] / wfm.upscale_f.y
print(len(bins))
assert (abs(bins[1] - bins[0]) - cam.upscale_f.y / cam.mdl["mask_deltay"]) < 1e-10


fig, ax = plt.subplots(1, 1, figsize=(6, 6))
fig.tight_layout()
ax.plot(bins, kernel)
ax.set_xlabel("detector bin [mm]", fontsize=12, fontweight='bold')
ax.set_ylabel("kernel value", fontsize=12, fontweight='bold')
ax.set_title("PSFY Kernel Profile", fontsize=14, pad=8, fontweight='bold')
ax.grid(visible=True, color="lightgray", linestyle="-", linewidth=0.3)
ax.tick_params(which='both', direction='in', width=2, length=7 if 'major' else 4)
ax.xaxis.set_ticks_position('both')
ax.yaxis.set_ticks_position('both')
plt.show()

In [ ]:
bins[:10], bins[-10:]